<a href="https://colab.research.google.com/github/PrairieResearchInstitute/2026_ml_eco_evo_collections_workshop/blob/main/structured_extraction/hands-on.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on — your own documents

Let's now apply the methods we learned in the workshop to papers and data you care about.

We will need the following information:
- **where your documents are** — PDFs, or photographs of labels and sheets
- a **system prompt** setting the LLM role
- a **user prompt** with details on WHAT to extract
- a **schema** constraining the shape of the output.
- the **model** we should use
- whether the model should use **thinking**
- the **context window** size

Then run the last cell and read the table.

The first code block is every function the workshop built,
collected in one place so you do not have to run four sessions again.

## Setup

We will start by setting up Ollama

In [ ]:
import glob, os, subprocess, sys, time

IN_COLAB = "google.colab" in sys.modules
REPO = "2026_ml_eco_evo_collections_workshop"
SESSION = "structured_extraction"       # this session's folder inside the repository
BRANCH = "main"   # switch to your working branch to test unmerged changes

if IN_COLAB:
    !DEBIAN_FRONTEND=noninteractive apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip -q install ollama pymupdf pandas pillow
    if os.path.basename(os.getcwd()) != SESSION:   # so this cell is safe to re-run
        if not os.path.isdir(REPO) and not os.path.isdir(SESSION):
            !git clone -q --branch {BRANCH} https://github.com/PrairieResearchInstitute/{REPO}.git
        os.chdir(SESSION if os.path.isdir(SESSION) else os.path.join(REPO, SESSION))
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import ollama

def server_ready(timeout=120):
    """Wait until Ollama answers, or give up."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            ollama.list()
            return True
        except Exception:
            time.sleep(1)
    return False

assert server_ready(), "Ollama did not start"
print("Ollama is up")

!ollama pull deepseek-ocr

## Everything the workshop built

Here we have all the imports, the constants and every function from
`workshop.ipynb`.

In [ ]:
import os, subprocess, sys

import time, base64, json, glob, re

import ollama, pymupdf, pandas as pd

from PIL import Image

from IPython.display import display, Image as ShowImage

import io

def server_ready(timeout=120):
    """Wait until Ollama answers, or give up."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            ollama.list()
            return True
        except Exception:
            time.sleep(1)
    return False

def gpu_report():
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                              "--format=csv,noheader"],
                             capture_output=True, text=True)
        print("GPU:", smi.stdout.strip() if smi.returncode == 0 else "none")
    except FileNotFoundError:
        print("GPU: none")
    try:
        ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
        rows = [r for r in ps.stdout.strip().splitlines()[1:] if r.strip()]
        print("Ollama is running:", rows or "nothing loaded yet")
    except FileNotFoundError:
        print("Ollama is running: the ollama command was not found -- "
              "did the setup cell above finish?")

CHAT_MODEL = "qwen3.5:4b"

OCR_MODEL  = "deepseek-ocr"

DEFAULT_DPI = 100

def open_pdf(path):
    return pymupdf.open(path)

def get_page_text(doc, page):
    """The text layer already stored inside the PDF. Free and instant."""
    if not 1 <= page <= doc.page_count:
        raise ValueError(f"page {page} out of range (1-{doc.page_count})")
    return doc[page - 1].get_text()

def render_page(doc, page, dpi=DEFAULT_DPI):
    """Draw a page as an image, encoded for sending to a model."""
    if not 1 <= page <= doc.page_count:
        raise ValueError(f"page {page} out of range (1-{doc.page_count})")
    return base64.b64encode(
        doc[page - 1].get_pixmap(dpi=dpi).tobytes("png")).decode()

from IPython.display import Image as ShowImage, display

def ocr_page(doc, page, dpi=DEFAULT_DPI):
    """Transcribe a page from its image, with region coordinates.

    Coordinates come back scaled 0-1000, labelled text / image /
    image_caption. No think= here: this model does not reason, it transcribes.
    """
    reply = ollama.generate(
        model=OCR_MODEL,
        prompt="<image>\n<|grounding|>Convert the document to markdown.",
        images=[render_page(doc, page, dpi=dpi)],
        options={"temperature": 0, "num_predict": 4096},
    )
    return reply.response or ""

def regions(ocr_text):
    """Every labelled region: [(label, x1, y1, x2, y2), ...], scaled 0-1000."""
    found = re.findall(r"(\w+)\s*\[\[\s*(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\s*\]\]",
                       ocr_text)
    return [(lab, *map(int, box)) for lab, *box in found]

def crop_region(doc, page, box, dpi=150, pad=0.01):
    """Cut one 0-1000 box out of a rendered page."""
    im = Image.open(io.BytesIO(
        doc[page - 1].get_pixmap(dpi=dpi).tobytes("png")))
    W, H = im.size
    _, x1, y1, x2, y2 = box
    return im.crop((int((x1/1000 - pad) * W), int((y1/1000 - pad) * H),
                    int((x2/1000 + pad) * W), int((y2/1000 + pad) * H)))

PNG_MODES = {"1", "L", "LA", "P", "RGB", "RGBA"}

def as_png_mode(image):
    """A version of an image that PNG can actually hold.

    Print-ready figures are often CMYK, and PNG has no CMYK: Pillow refuses to
    write it rather than approximating. Anything PNG cannot represent becomes
    RGB here, once, so that saving it and sending it both work later.
    """
    return image if image.mode in PNG_MODES else image.convert("RGB")

MIN_FIGURE_AREA = 0.03

MAX_FIGURE_AREA = 0.90

def ocr_elements(ocr_text):
    """The OCR model's regions, in the order it read them: (label, box, text)."""
    marks = list(re.finditer(
        r"(\w+)\s*\[\[\s*(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\s*\]\]", ocr_text))
    found = []
    for i, mark in enumerate(marks):
        stop = marks[i + 1].start() if i + 1 < len(marks) else len(ocr_text)
        box = ("region", *(int(mark.group(k)) for k in range(2, 6)))
        found.append((mark.group(1), box, ocr_text[mark.end():stop].strip()))
    return found

def page_elements(doc, page, transcript=None, dpi=150):
    """A page as ("text", str) and ("figure", Image) pieces, in reading order."""
    if transcript is not None:                    # a scan: already in order
        pieces = []
        for label, box, text in ocr_elements(transcript):
            if label == "image":
                pieces.append(("figure", crop_region(doc, page, box, dpi=dpi)))
            elif text:
                pieces.append(("text", text))
        return pieces

    pg = doc[page - 1]                            # born-digital: sort by
    page_area = pg.rect.width * pg.rect.height    # where things sit
    placed = []
    for x0, y0, x1, y1, text, _, kind in pg.get_text("blocks", sort=True):
        if kind == 0 and text.strip():
            placed.append((y0, ("text", text.strip())))
    for xref, *_ in pg.get_images(full=True):
        rects = pg.get_image_rects(xref)
        if not rects:
            continue
        share = max((r.width * r.height) / page_area for r in rects)
        if not MIN_FIGURE_AREA <= share <= MAX_FIGURE_AREA:
            continue
        # extract_image gives the bytes as stored, whatever the colour depth:
        # line art at 1 bit per pixel comes back as readily as a photograph.
        figure = as_png_mode(Image.open(io.BytesIO(doc.extract_image(xref)["image"])))
        placed.append((rects[0].y0, ("figure", figure)))
    return [piece for _, piece in sorted(placed, key=lambda item: item[0])]

def show_elements(pieces):
    for kind, value in pieces:
        if kind == "text":
            print(f"  text   {' '.join(value.split())[:72]}")
        else:
            print(f"  figure {value.size[0]}x{value.size[1]}")

def process_pdf(path, out_dir="processed", needs_ocr=False):
    """Turn a PDF into a folder of markdown pages you can open and read."""
    name = os.path.splitext(os.path.basename(path))[0]
    folder = os.path.join(out_dir, name)
    os.makedirs(os.path.join(folder, "figures"), exist_ok=True)

    doc = open_pdf(path)
    for page in range(1, doc.page_count + 1):
        page_file = os.path.join(folder, f"page-{page:03d}.md")
        if os.path.exists(page_file):
            continue                      # done on an earlier run

        transcript = ocr_page(doc, page) if needs_ocr else None
        lines, figures = [], 0
        for kind, value in page_elements(doc, page, transcript):
            if kind == "text":
                lines.append(value)
            else:
                figures += 1
                relative = f"figures/p{page:03d}-fig{figures:02d}.png"
                value.save(os.path.join(folder, relative))
                lines.append(f"![figure]({relative})")
        with open(page_file, "w") as fh:
            fh.write(f"# page {page}\n\n" + "\n\n".join(lines) + "\n")
        print(f"  page {page}: {figures} figure(s)")
    return folder

def load_pages(folder, pages=None):
    """A processed folder back as (text, figures), ready to send to a model.

    Ollama attaches images to a message rather than placing them in the text,
    so each figure link becomes a numbered marker where it stood, and the
    figures are handed over in that same order.
    """
    wanted = None if pages is None else {int(p) for p in pages}
    chunks, figures = [], []

    for page_file in sorted(glob.glob(os.path.join(folder, "page-*.md"))):
        page = int(os.path.basename(page_file)[5:8])
        if wanted is not None and page not in wanted:
            continue
        with open(page_file) as fh:
            text = fh.read()

        def mark(match):
            figures.append(Image.open(os.path.join(folder, match.group(1))))
            return f"[figure {len(figures)} appears here]"

        chunks.append(re.sub(r"!\[[^\]]*\]\(([^)]+)\)", mark, text))
    return "\n\n".join(chunks), figures

SPECIES_SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "author": {"type": "string"},
        "is_new": {"type": "boolean"},
        "max_length": {"type": "number"},
        "n_photos": {"type": "integer"},
        "photo_alive": {"type": "boolean"},
        "similar_species": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["name", "author"],
    "additionalProperties": False
}

def parse_json(text):
    """The JSON in a reply, even if the model wrapped it in something else."""
    text = (text or "").strip()
    if not text:
        raise ValueError("the model returned nothing to parse -- it may have "
                         "spent the whole reply reasoning")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end <= start:
            raise ValueError(f"no JSON in the reply: {text[:120]!r}") from None
        return json.loads(text[start:end + 1])

PAPER_SCHEMA = {
    "type": "object",
    "properties": {
        "species": {"type": "array", "items": SPECIES_SCHEMA}
    },
    "required": ["species"],
    "additionalProperties": False,
}

def as_image_data(fig, max_side=1024):
    """A figure, shrunk if it is huge, encoded the way a model wants it."""
    small = as_png_mode(fig).copy()
    small.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    small.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

WHOLE_PAPER_PROMPT = (
    '''List every species this paper describes, using both the text and the figures.
    name is the species name.
    author is the taxonomic authority for that name -- the person who described it -- not the author of this paper; for a species described as new here, use the paper's own authors.
    max_length is the largest body length in mm given for the species.
    is_new is whether the species is described here
    n_photos is the number of photos used to illustrate the species. Photos only, no illustrations.
    photo_alive is whether any photo shows the insect alive in its habitat
    similar_species are all species that the text mentions are similar
    Do not invent values that are not stated.

    '''
)

def extract(text, schema=PAPER_SCHEMA, prompt=WHOLE_PAPER_PROMPT, figures=None,
            system=None, think=False, model=CHAT_MODEL, client=None,
            num_ctx=16384, schema_in_prompt=False):
    """Text (and figures) in, structured data out.

    client is None for the server on this machine. A client pointed at Ollama's
    servers instead is the only change needed to run any of this on a far
    bigger model.

    system sets the role the model should answer in, the way session 1 did.

    think is the choice we just made by hand: reasoning before answering,
    which helps the fields that need judgement and costs time and context.

    schema_in_prompt writes the schema into the prompt as well as passing it as
    format=. Locally that is redundant -- format= is enforced. Ollama's cloud
    does not enforce it, so there asking is all you have.
    """
    if schema_in_prompt:
        JSON_ONLY = (
         "\n\nReply with JSON only -- no prose, no markdown, no code fence -- "
         "matching exactly this schema:\n{schema}\n\n"
        )
        prompt = prompt + JSON_ONLY.format(schema=json.dumps(schema, indent=1))

    # Roughly four characters to a token, and roughly a thousand tokens an
    # image. Reasoning is written into the same window and cannot be capped, so
    # when it is on we set a few thousand tokens aside for it. Ollama drops
    # whatever will not fit without saying so, which is the one failure you
    # cannot see in the answer, so say it here.
    budget = (len(prompt) + len(text)) // 4 + 1000 * len(figures or [])
    if think:
        budget += 4000
    if budget > num_ctx:
        room = " (including room set aside for reasoning)" if think else ""
        print(f"WARNING: about {budget:,} tokens for a context of {num_ctx:,}"
              f"{room}. The overflow will be dropped silently. Raise num_ctx, "
              f"send fewer pages, or send fewer figures.")

    message = {"role": "user", "content": prompt + text}
    if figures:
        message["images"] = [as_image_data(fig) for fig in figures]

    messages = [{"role": "system", "content": system}] if system else []
    messages.append(message)

    chat = (client or ollama).chat
    reply = chat(
        model=model,
        messages=messages,
        format=schema,
        think=think,
        options={"temperature": 0, "num_ctx": num_ctx},
    )
    return parse_json(reply.message.content)

def to_table(data, key="species"):
    """The array of records inside an extraction result, as a table.

    Our schemas wrap the records in one named array -- "species" in
    PAPER_SCHEMA. Pass key= if you named yours something else.

    Anything the schema asks for beside that array -- a fact about the document
    rather than about one record -- becomes a column of its own, repeated down
    the rows, so it travels with the records it belongs to.
    """
    rows = pd.DataFrame(data.get(key, []))
    for field, value in data.items():
        if field == key:
            continue
        rows[field] = (json.dumps(value) if isinstance(value, (list, dict))
                       else value)
    return rows

TOOLS = [
    {"type": "function",
     "function": {
         "name": "get_page_text",
         "description": ("Return the PDF's embedded text layer for a page. "
                         "Free and instant, but on scanned documents it can be "
                         "poor-quality OCR with garbled words."),
         "parameters": {"type": "object",
                        "properties": {"page": {"type": "integer",
                                                "description": "1-based page number"}},
                        "required": ["page"]}}},
    {"type": "function",
     "function": {
         "name": "ocr_page",
         "description": ("Re-read a page from its image with a dedicated OCR "
                         "model. Slower, far more accurate on scans."),
         "parameters": {"type": "object",
                        "properties": {"page": {"type": "integer",
                                                "description": "1-based page number"}},
                        "required": ["page"]}}},
]

def run_tool_loop(messages, tools, impls, think=True, max_turns=6):
    """Let the model call tools until it has an answer.

    Returns (answer, calls_it_made).
    """
    messages = list(messages)
    calls = []
    for _ in range(max_turns):
        reply = ollama.chat(model=CHAT_MODEL, messages=messages, tools=tools,
                            think=think, options={"temperature": 0})
        msg = reply.message
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": msg.tool_calls or []})
        if not msg.tool_calls:
            return (msg.content or ""), calls
        for call in msg.tool_calls:
            name = call.function.name
            args = dict(call.function.arguments)
            calls.append((name, args))
            try:
                result = impls[name](**args)
            except Exception as exc:
                result = f"ERROR: {exc}"        # tell the model, don't crash
            messages.append({"role": "tool", "name": name,
                             "content": str(result)[:6000]})
    return f"stopped after {max_turns} turns", calls

## 1. Your documents

Click the folder icon 📁 in the left sidebar and upload **two to five documents**
into `my_pdfs/`. Start small: we now want to try things out, not write a paper.

Despite the name of the folder, feel free to add both pdfs and image files (e.g. from specimen labels):

- **PDFs** (`.pdf`) — what we have seem so far: many pages, contains images and a hidden text layer that may or may not be accurate.
- **Photos** (`.jpg`, `.png`, `.tif`) — a specimen label, a herbarium sheet, or a
  page you photographed. There is no text, so we will use them as images.

Run the cell below. For PDFs it prints a sample of the stored text so we can
judge its quality; for photos it prints the size.

In [ ]:
MY_FOLDER = "my_pdfs"          # where your documents are

PDF_SUFFIXES   = (".pdf",)
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

def find_files(folder, suffixes):
    """Files of these kinds in a folder, ignoring case: cameras write .JPG."""
    if not os.path.isdir(folder):
        return []
    return sorted(os.path.join(folder, name) for name in os.listdir(folder)
                  if name.lower().endswith(suffixes) and not name.startswith("."))

os.makedirs(MY_FOLDER, exist_ok=True)
my_papers = find_files(MY_FOLDER, PDF_SUFFIXES)
my_images = find_files(MY_FOLDER, IMAGE_SUFFIXES)

if not my_papers and not my_images:
    print(f"Nothing in {MY_FOLDER}/ yet. Upload PDFs or photos with the folder "
          f"icon in the sidebar, then run this cell again.")

for path in my_papers:
    doc = open_pdf(path)
    sample = get_page_text(doc, min(2, doc.page_count))[:400]
    print(f"=== {os.path.basename(path)} ({doc.page_count} pages) ===")
    print(sample.strip() or "(no text at all -- this one is certainly a scan)")
    print()

for path in my_images:
    with Image.open(path) as im:
        print(f"=== {os.path.basename(path)} (photo, {im.size[0]}x{im.size[1]}) ===")
        print("read from the image; nothing to judge here")
        print()

### Which ones do you not trust?

List the files whose text looked wrong. Those will get re-read page by page with the
OCR model.

Leave the list empty if they all looked good.

We could use the AI equipped with tools to decide, but human intelligence will be faster in this case with just a few PDFs that you chose.

This step is about PDFs only. A photo has no stored text to distrust, so it is
always read from the image — the next cell is where you choose how.

In [ ]:
RESCAN = [
    # "the_1929_scan.pdf",       # <- file names, one per line
]

my_folders = {}
rescanned = False
for path in my_papers:
    name = os.path.basename(path)
    doc = open_pdf(path)
    empty = len(get_page_text(doc, min(2, doc.page_count)).strip()) < 100
    needs_ocr = name in RESCAN or empty
    rescanned = rescanned or needs_ocr
    print(f"{name}: {'re-reading every page' if needs_ocr else 'using the stored text'}"
          f"{' (nothing there to use)' if empty and name not in RESCAN else ''}")
    my_folders[path] = process_pdf(path, needs_ocr=needs_ocr)

if not my_papers:
    print("No PDFs here, so nothing to split into pages.")

if rescanned:
    # Hand the GPU back. Ollama keeps a model loaded for a few minutes after its
    # last call, and deepseek-ocr's 6.7 GB is 6.7 GB the extraction below cannot
    # use for its context -- or for a bigger MY_MODEL, which will not sit
    # alongside it at all.
    ollama.generate(model=OCR_MODEL, prompt="", keep_alive=0)
    print(f"\nreleased {OCR_MODEL}")

if my_folders:
    print("\nwritten to:", *my_folders.values(), sep="\n  ")

### Photos: read by the model, or transcribed first?

Ignore this section if you have pdf document input, this applies only to label transcription.

One thing to consider is that here we limit the size of the input to our chat model by reducing photo resolutions to 1024 pixels at most. If your labels are not cropped and the text is present in only a small section, it might be better to first do an OCR pass a full resolution and then feed to the model that will do the extraction.

So, if your characters are very small compared to the photo size, you have two alternatives:

- Crop the labels tighter before you upload them. Free, and it helps most.
- Set `MY_OCR_LABELS = True` below to send each photo to `deepseek-ocr` first at
  `MY_OCR_SIDE` pixels, and hand the extraction model that transcript *and* the
  image.

Leave it `False` for a first run: let's see what the chat model does on
its own before you decide it needs help.

In [ ]:
MY_OCR_LABELS = False    # transcribe photos with deepseek-ocr before extracting?
MY_OCR_SIDE   = 4000     # pixels on the longest side sent to the OCR model

my_transcripts = {}
if my_images and MY_OCR_LABELS:
    for path in my_images:
        reply = ollama.generate(
            model=OCR_MODEL,
            prompt="<image>\n<|grounding|>Convert the document to markdown.",
            images=[as_image_data(Image.open(path), max_side=MY_OCR_SIDE)],
            options={"temperature": 0, "num_predict": 4096},
        )
        raw = reply.response or ""
        # ocr_elements drops the 0-1000 coordinate markers and keeps the words,
        # in the order the model read them.
        text = "\n\n".join(t for _, _, t in ocr_elements(raw) if t) or raw
        my_transcripts[path] = text
        print(f"{os.path.basename(path)}: {len(text)} characters  "
              f"{' '.join(text.split())[:64]}")

    ollama.generate(model=OCR_MODEL, prompt="", keep_alive=0)   # hand the GPU back
    print(f"\nreleased {OCR_MODEL}")
elif my_images:
    print(f"{len(my_images)} photo(s) will be read straight from the image.")

## 2. The system prompt

Who should the model be while it reads? This is the role, not the task —
session 1's point that these are role-playing machines. Be specific about the
field and about care: a model told it is a careful taxonomist behaves
differently from one told nothing.

In [ ]:
MY_SYSTEM = (
    '''You are a careful DESCRIBE THE SPECIALITY HERE.
    You read primary literature and record only what the text actually says.
    When something is not stated, you leave it out rather than guessing.'''
)

## 3. The user prompt

What do you want out of each document? Say what to extract, and say what every
field of your schema means.

If you do not explain a field, you will let the LLM judge, and it can make more mistakes.

In [ ]:
MY_PROMPT = (
    '''DESCRIBE WHAT TO EXTRACT HERE.
    FIELD_ONE is ...
    FIELD_TWO is ...
    FIELD_THREE is ...
    Do not invent values that are not stated in the text.'''
)

## 4. The schema

Now constrain the fields you want, and their types. Use two levels, as in session 3: one for records, with possibly many records per document, and another for document.

Valid types are `"string"`, `"number"`, `"integer"`, `"boolean"`, `"object"` and `"array"`. Put
in `required` only the fields that must always be there.

Use `"additionalProperties": True` only if you want to allow the model to find properties that you did not list explicitly.

The two levels hold for photos too, and are worth having even when a photo shows
one label: a record is one label, or one specimen, and the document-level fields
are facts about the photograph — the drawer or unit tray it came from, who
imaged it, how many labels are visible.

**Tip:** writing schemas by hand is fiddly. You can start the broad strokes by hand and then get Claude or ChatGPT
to check whether JSON schema is valid and fix it. You can also try Gemini on this notebook.

In [ ]:
MY_ITEM_SCHEMA = {                 # one record
    "type": "object",
    "properties": {
        "FIELD_ONE":   {"type": "string"},
        "FIELD_TWO":   {"type": "string"},
        "FIELD_THREE": {"type": "number"},
    },
    "required": ["FIELD_ONE"],
    "additionalProperties": False,
}

MY_SCHEMA = {                      # a document holds many records, and document-level attributes
    "type": "object",
    "properties": {
        "records": {"type": "array", "items": MY_ITEM_SCHEMA},
        "DOCUMENT_ATTRIBUTE_ONE": {"type": "number"}
    },
    "required": ["records"],
    "additionalProperties": False,
}

## 5. How it should read

Three settings, and they pull on each other.

**Model.** [`qwen3.5:4b`](https://ollama.com/library/qwen3.5) is the default: a
middle size at 3.4 GB, quick to download and quick to answer, and steadier than
the 2B at holding to a schema. If you have the memory, try
[`qwen3.5:9b`](https://ollama.com/library/qwen3.5/tags) (6.6 GB) or
[`gemma3:12b`](https://ollama.com/library/gemma3) (8.1 GB) and compare — both
read images, which this pipeline needs, and both will likely be more accurate on
your documents than the 4B.

Name it below and the cell after that pulls it for you. Remember the larger
ones take room the key-value cache was using, so a bigger model may mean a
smaller `MY_NUM_CTX`. `!ollama list` shows what this machine already holds.

**Thinking.** Does what you are extracting call for judgement, or is it copying
values that sit plainly on the page? You saw in session 3 which fields reasoning
helped and which it left alone.

**Context.** How much the model may hold at once. It has to fit your document,
its figures, and the reasoning as well if you turned that on — the run cell
warns you when it will not.

In [ ]:
MY_MODEL = CHAT_MODEL   # "qwen3.5:4b" by default.
                        # To try another, name it here, like:
                        # MY_MODEL = "gemma3:12b"
                        # and the next cell pulls it for you.

MY_THINK = False        # reason before answering? Slower, and the reasoning is
                        # use context window along with everything
                        # else -- so if you turn this on, check MY_NUM_CTX too.

MY_NUM_CTX = 65_536     # how much the model may read at once. Adjust up if your
                        # papers are too big, but uses more memory.

In [ ]:
# Pull whatever you chose above. Ollama skips the download if it already has
# the model, so this is quick on a re-run and slow the first time you name a
# new one.
!ollama pull {MY_MODEL}

## 6. Run it

Each document (PDF split into pages, or a photo on its own) goes to the model
with your system prompt, your user prompt and your schema. What comes back is one
JSON object per document, in the shape the schema asked for. The workhorse is the function `extract()`
defined above, which includes the LLM call.

We save the JSON as it arrives, keeping nested fields — an array inside a record, an object inside
that, etc.

In [ ]:
# Everything found in step 1, as (name, text, figures) -- a processed PDF is
# pages of text plus its figures, a photo is the image and whatever the OCR
# model made of it, and extract() cannot tell the difference.
documents = []
for path, folder in my_folders.items():
    text, figures = load_pages(folder)
    documents.append((os.path.basename(path), text, figures))
for path in my_images:
    documents.append((os.path.basename(path), my_transcripts.get(path, ""),
                      [Image.open(path)]))

class Recorder:
    """A stand-in for the ollama module that keeps every reply.

    extract() returns the parsed JSON and drops the rest, and the reasoning is
    in the rest. It takes client= so the same code can run against Ollama's
    cloud; that seam lets us hold on to the replies here, for step 8.
    """
    def __init__(self):
        self.name, self.replies = None, {}

    def chat(self, **kwargs):
        reply = ollama.chat(**kwargs)
        self.replies[self.name] = reply
        return reply

recorder = Recorder()
my_data = {}                   # document name -> the JSON the model returned

if not documents:
    print(f"Upload documents into {MY_FOLDER}/ and run step 1 first.")

for name, text, figures in documents:
    print(f"{name}: {len(text)} characters, {len(figures)} figures")
    recorder.name = name
    try:
        my_data[name] = extract(text, schema=MY_SCHEMA, prompt=MY_PROMPT,
                                figures=figures, system=MY_SYSTEM,
                                model=MY_MODEL, think=MY_THINK,
                                num_ctx=MY_NUM_CTX, client=recorder)
    except Exception as exc:
        print(f"   FAILED -- {exc}")

if my_data:
    with open("my_results.json", "w") as fh:
        json.dump(my_data, fh, indent=2, ensure_ascii=False)
    print(f"\n{len(my_data)} document(s), saved to my_results.json\n")

    shown = json.dumps(my_data, indent=2, ensure_ascii=False)
    print(shown[:4000])
    if len(shown) > 4000:
        print(f"\n[...{len(shown) - 4000:,} more characters -- all of it is in "
              f"my_results.json]")

## 7. The table

Depending on the application you have for your data, a table may be more convenient, but it collapses arrays and objects to one field (i.e. it flattens the data). But here we will show how to convert JSON to a table with one row per record and a `source` column saying which
document it came from.

Now **read the table against the documents**. Because we have a schema, the
shape of the output is guaranteed. But the content may be wrong. How would you
measure how much to trust?

In [ ]:
frames = []
for name, data in my_data.items():
    rows = to_table(data, key="records")     # pass the name your schema used
    print(f"{name}: {len(rows)} record(s)")
    if len(rows):
        rows.insert(0, "source", name)
        frames.append(rows)

my_table = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
my_table.to_csv("my_results.csv", index=False)
print(f"\n{len(my_table)} rows, saved to my_results.csv")
display(my_table)

## 8. What the model was thinking

Thinking is helpful for complex problems and we typically do not save it, just the response. However, here we will have a look at what the model output as thinking so you have some idea of what it looks like.

If `MY_THINK` is `False` in step 5, there is
nothing here to print.

Read it when a value came out wrong and you cannot see why. The reasoning
usually shows which of the three it was: the model misread the page, it ignored
your description of the field, or it was guessing.

Knowing this can help you design better prompts and schemas to make your request more clear with less room for error.

In [ ]:
for name, reply in recorder.replies.items():
    thinking = (getattr(reply.message, "thinking", None) or "").strip()
    print(f"=== {name} ===")
    print(thinking or "(no reasoning returned -- MY_THINK is off, or this "
                      "model does not reason)")
    print()